# FLAME Mesh Repair & Reconstruction Service

This notebook provides a GPU-accelerated mesh repair pipeline using FLAME 3D face model.

**Modes:**
- `repair_only`: Basic mesh repair (fix normals, fill holes, remove degenerate faces)
- `flame_fit`: Fit FLAME model to input mesh, output clean FLAME topology
- `flame_blend`: Blend original mesh with FLAME fitting result

In [2]:
!pip install -q torch numpy trimesh scipy open3d tqdm chumpy pyrender

In [3]:
!git clone https://github.com/kk20250706/demo2026.git 2>/dev/null || true
import sys
sys.path.insert(0, '/content/demo2026')

In [4]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 1. Upload Files

Upload your FBX mesh file and (optionally) the FLAME model file (`generic_model.pkl`).

Download FLAME model from: https://flame.is.tue.mpg.de/download.php

In [6]:
print("Upload generic_model.pkl (required for flame_blend mode):")
uploaded_model = files.upload()
model_name = list(uploaded_model.keys())[0]
with open(f"models/{model_name}", "wb") as f:
    f.write(uploaded_model[model_name])
print(f"FLAME model saved to models/{model_name}")


Upload generic_model.pkl (required for flame_blend mode):


Saving generic_model.pkl to generic_model.pkl
FLAME model saved to models/generic_model.pkl


In [5]:
!apt-get install -y -qq blender > /dev/null 2>&1 && echo "Blender installed"


Blender installed


In [6]:
import numpy as np
np.bool = np.bool_
np.int = np.int_
np.float = np.float64
np.complex = np.complex128
np.object = np.object_
np.unicode = np.str_
np.str = np.str_

import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec


In [5]:
import sys
sys.path.insert(0, '/content/demo2026')


In [9]:
from pathlib import Path

p = Path('/content/demo2026/flame_repair/flame_model.py')
text = p.read_text()

old = '    T = torch.einsum("bvj,bjmn->bvmn", weights, joint_transforms)\n'
new = '    weights_batch = weights.unsqueeze(0).expand(batch_size, -1, -1)\n    T = torch.einsum("bvj,bjmn->bvmn", weights_batch, joint_transforms)\n'

if old in text:
    text = text.replace(old, new)
    p.write_text(text)
    print("patched flame_model.py")
else:
    print("target line not found, maybe already patched")


patched flame_model.py


In [10]:
import sys
sys.path.insert(0, '/content/demo2026')


In [11]:
from flame_repair.pipeline import MeshRepairPipeline

MODE = 'flame_blend'
SMOOTH_ITERS = 2
FLAME_ITERS = 500
FLAME_MODEL_PATH = 'models/generic_model.pkl'

output_path = f'output/repaired_{input_filename}'
flame_path = FLAME_MODEL_PATH if MODE != 'repair_only' else None
pipeline = MeshRepairPipeline(flame_model_path=flame_path, device='auto')

result_mesh = pipeline.run(
    input_path=input_path,
    output_path=output_path,
    mode=MODE,
    smooth_iters=SMOOTH_ITERS,
    flame_iters=FLAME_ITERS,
)



  Mesh Repair Pipeline
  Input:  input/liukui.fbx
  Output: output/repaired_liukui.fbx
  Mode:   flame_blend

[IO] Converting FBX -> OBJ via Blender...
[IO] Converted to input/liukui.obj
[IO] Loaded mesh: 22720 vertices, 43118 faces

[Step 1] Basic Mesh Repair
[Repair] Vertices: 22720 -> 21631
[Repair] Faces: 43118 -> 43118
[Repair] Watertight: False

[Step 2] Laplacian Smoothing (iterations=2)

[Step 4] FLAME Fitting & Blending
[FLAME] Using device: cuda
[FLAME] Fitting to target mesh (21631 vertices)...


FLAME Fitting:   0%|          | 0/500 [00:00<?, ?it/s]


RuntimeError: einsum(): the number of subscripts in the equation (3) does not match the number of dimensions (2) for operand 0 and no ellipsis was given

## 2. Run Mesh Repair

In [21]:
from flame_repair.pipeline import MeshRepairPipeline

#@markdown ### Configuration
MODE = 'flame_fit'  #@param ['repair_only', 'flame_fit', 'flame_blend']
SMOOTH_ITERS = 5  #@param {type: 'integer'}
FLAME_ITERS = 500  #@param {type: 'integer'}
FLAME_MODEL_PATH = 'models/generic_model.pkl'  #@param {type: 'string'}
input_filename = 'liukui.fbx'
input_path = 'input/liukui.fbx'
output_path = f'output/repaired_{input_filename}'

flame_path = FLAME_MODEL_PATH if MODE != 'repair_only' else None
pipeline = MeshRepairPipeline(flame_model_path=flame_path, device='auto')

result_mesh = pipeline.run(
    input_path=input_path,
    output_path=output_path,
    mode=MODE,
    smooth_iters=SMOOTH_ITERS,
    flame_iters=FLAME_ITERS,
)


  Mesh Repair Pipeline
  Input:  input/liukui.fbx
  Output: output/repaired_liukui.fbx
  Mode:   repair_only

[IO] Converting FBX -> OBJ via Blender...
[IO] Converted to input/liukui.obj
[IO] Loaded mesh: 22720 vertices, 43118 faces

[Step 1] Basic Mesh Repair
[Repair] Vertices: 22720 -> 21631
[Repair] Faces: 43118 -> 43118
[Repair] Watertight: False

[Step 2] Laplacian Smoothing (iterations=5)
[IO] Converting OBJ -> FBX via Blender...
[IO] Exported to output/repaired_liukui.fbx
[IO] Saved mesh to output/repaired_liukui.fbx: 21631 vertices, 43118 faces

[Done] Output saved to output/repaired_liukui.fbx


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Visualize Result

In [ ]:
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def plot_mesh(mesh, title='Mesh', max_faces=5000):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    verts = mesh.vertices
    faces = mesh.faces
    if len(faces) > max_faces:
        idx = np.random.choice(len(faces), max_faces, replace=False)
        faces = faces[idx]
    ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2],
                    triangles=faces, alpha=0.6, edgecolor='gray', linewidth=0.1)
    ax.set_title(f'{title} ({len(mesh.vertices)} verts, {len(mesh.faces)} faces)')
    plt.tight_layout()
    plt.show()

plot_mesh(result_mesh, 'Repaired Mesh')

In [14]:
!cd /content/demo2026 && git fetch origin demo1 && git reset --hard origin/demo1


remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 905 bytes | 452.00 KiB/s, done.
From https://github.com/kk20250706/demo2026
 * branch            demo1      -> FETCH_HEAD
   083bfb7..8f86b04  demo1      -> origin/demo1
HEAD is now at 8f86b04 fix: use legacy Blender OBJ export API for Colab compatibility


## 4. Download Result

In [ ]:
import glob

output_files = glob.glob('output/*')
for f in output_files:
    print(f'Downloading: {f}')
    files.download(f)

In [12]:
import sys
sys.path.insert(0, '/content/demo2026')

import torch
import importlib
import flame_repair.flame_model as fm

def patched_lbs(v, J, rot_mats, parents, weights):
    batch_size = v.shape[0]
    num_joints = J.shape[1]

    transforms = torch.zeros(batch_size, num_joints, 4, 4, device=v.device, dtype=v.dtype)

    for i in range(num_joints):
        rot = rot_mats[:, i]
        t = J[:, i].unsqueeze(-1)
        pad = torch.zeros(batch_size, 1, 4, device=v.device, dtype=v.dtype)
        pad[:, :, 3] = 1
        transform = torch.cat([torch.cat([rot, t], dim=-1), pad], dim=1)

        if parents[i] < 0:
            transforms[:, i] = transform
        else:
            transforms[:, i] = torch.bmm(transforms[:, parents[i]].clone(), transform)

    joint_transforms = transforms.clone()
    joint_rest = torch.cat(
        [J, torch.zeros(batch_size, num_joints, 1, device=v.device, dtype=v.dtype)],
        dim=-1,
    ).unsqueeze(-1)
    joint_transforms[:, :, :, 3:] = joint_transforms[:, :, :, 3:] - torch.matmul(joint_transforms, joint_rest)

    weights_batch = weights.unsqueeze(0).expand(batch_size, -1, -1)
    T = torch.einsum("bvj,bjmn->bvmn", weights_batch, joint_transforms)

    v_homo = torch.cat(
        [v, torch.ones(batch_size, v.shape[1], 1, device=v.device, dtype=v.dtype)],
        dim=-1,
    )
    v_posed = torch.matmul(T, v_homo.unsqueeze(-1))[:, :, :3, 0]
    return v_posed

fm.lbs = patched_lbs
print("patched flame_repair.flame_model.lbs")


patched flame_repair.flame_model.lbs


In [13]:
import sys
sys.path.insert(0, '/content/demo2026')
import flame_repair.flame_model as fm
import inspect
print(inspect.getsource(fm.lbs))


def patched_lbs(v, J, rot_mats, parents, weights):
    batch_size = v.shape[0]
    num_joints = J.shape[1]

    transforms = torch.zeros(batch_size, num_joints, 4, 4, device=v.device, dtype=v.dtype)

    for i in range(num_joints):
        rot = rot_mats[:, i]
        t = J[:, i].unsqueeze(-1)
        pad = torch.zeros(batch_size, 1, 4, device=v.device, dtype=v.dtype)
        pad[:, :, 3] = 1
        transform = torch.cat([torch.cat([rot, t], dim=-1), pad], dim=1)

        if parents[i] < 0:
            transforms[:, i] = transform
        else:
            transforms[:, i] = torch.bmm(transforms[:, parents[i]].clone(), transform)

    joint_transforms = transforms.clone()
    joint_rest = torch.cat(
        [J, torch.zeros(batch_size, num_joints, 1, device=v.device, dtype=v.dtype)],
        dim=-1,
    ).unsqueeze(-1)
    joint_transforms[:, :, :, 3:] = joint_transforms[:, :, :, 3:] - torch.matmul(joint_transforms, joint_rest)

    weights_batch = weights.unsqueeze(0).expand(batch_siz

In [15]:
import sys
sys.path.insert(0, '/content/demo2026')

import torch
import flame_repair.flame_model as fm

def patched_lbs(v, J, rot_mats, parents, weights):
    batch_size = v.shape[0]
    num_joints = J.shape[1]

    transforms_list = []

    for i in range(num_joints):
        rot = rot_mats[:, i]
        t = J[:, i].unsqueeze(-1)

        upper = torch.cat([rot, t], dim=-1)
        lower = torch.tensor([0.0, 0.0, 0.0, 1.0], device=v.device, dtype=v.dtype)
        lower = lower.view(1, 1, 4).expand(batch_size, -1, -1)
        transform = torch.cat([upper, lower], dim=1)

        if parents[i] < 0:
            current = transform
        else:
            current = torch.bmm(transforms_list[int(parents[i])], transform)

        transforms_list.append(current)

    transforms = torch.stack(transforms_list, dim=1)

    joint_rest = torch.cat(
        [J, torch.zeros(batch_size, num_joints, 1, device=v.device, dtype=v.dtype)],
        dim=-1,
    ).unsqueeze(-1)

    joint_offsets = torch.matmul(transforms, joint_rest)
    joint_transforms = transforms.clone()
    joint_transforms = torch.cat(
        [joint_transforms[:, :, :, :3], joint_transforms[:, :, :, 3:] - joint_offsets],
        dim=-1,
    )

    weights_batch = weights.unsqueeze(0).expand(batch_size, -1, -1)
    T = torch.einsum("bvj,bjmn->bvmn", weights_batch, joint_transforms)

    v_homo = torch.cat(
        [v, torch.ones(batch_size, v.shape[1], 1, device=v.device, dtype=v.dtype)],
        dim=-1,
    )

    v_posed = torch.matmul(T, v_homo.unsqueeze(-1))[:, :, :3, 0]
    return v_posed

fm.lbs = patched_lbs
print("patched flame_repair.flame_model.lbs (no inplace ops)")


patched flame_repair.flame_model.lbs (no inplace ops)
